In [0]:
%sql

DESCRIBE TABLE catalog_au.golden.dim_entidad;

In [0]:
%sql

SELECT
    COUNT(*) AS total_registros
FROM catalog_au.golden.dim_entidad;

In [0]:
%sql

SHOW CREATE TABLE catalog_au.golden.dim_entidad;

In [0]:
%sql

SELECT
    codigo_entidad,
    MAX(CASE WHEN anio = 2024 THEN nombre_entidad END)
        AS nombre_2024,
    MAX(CASE WHEN anio = 2025 THEN nombre_entidad END)
        AS nombre_2025,
    MAX(CASE WHEN anio = 2026 THEN nombre_entidad END)
        AS nombre_2026,
    COUNT(DISTINCT anio) AS anios_presentes
FROM catalog_au.silver.sfc_indicadores_v2
GROUP BY codigo_entidad
ORDER BY codigo_entidad;

In [0]:
%sql

SELECT
    codigo_entidad,
    COUNT(DISTINCT TRIM(nombre_entidad)) AS nombres_distintos,
    COLLECT_SET(TRIM(nombre_entidad)) AS nombres_encontrados
FROM catalog_au.silver.sfc_indicadores_v2
GROUP BY codigo_entidad
HAVING COUNT(DISTINCT TRIM(nombre_entidad)) > 1
ORDER BY codigo_entidad;

In [0]:
%sql

WITH entidades AS (
    SELECT DISTINCT
        codigo_entidad,
        nombre_entidad,
        SPLIT(codigo_entidad, '-')[4] AS segmento_entidad
    FROM catalog_au.silver.sfc_indicadores_v2
)

SELECT
    CASE
        WHEN segmento_entidad <> '0'
            THEN 'ENTIDAD_INDIVIDUAL'
        ELSE 'AGRUPACION'
    END AS clasificacion,
    COUNT(DISTINCT codigo_entidad) AS total_codigos
FROM entidades
GROUP BY 1
ORDER BY 1;

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_entidades_base = (
    spark.table("catalog_au.silver.sfc_indicadores_v2")
    .select(
        "codigo_entidad",
        "nombre_entidad",
        "fecha_corte"
    )
    .withColumn(
        "nombre_normalizado",
        F.upper(
            F.regexp_replace(
                F.trim(F.col("nombre_entidad")),
                r"\s+",
                " "
            )
        )
    )
    .dropDuplicates([
        "codigo_entidad",
        "nombre_normalizado",
        "fecha_corte"
    ])
)

ventana = Window.partitionBy(
    "codigo_entidad"
).orderBy("fecha_corte")

df_entidades_cambios = (
    df_entidades_base
    .withColumn(
        "nombre_anterior",
        F.lag("nombre_normalizado").over(ventana)
    )
    .withColumn(
        "es_cambio",
        F.when(
            F.col("nombre_anterior").isNull() |
            (
                F.col("nombre_normalizado") !=
                F.col("nombre_anterior")
            ),
            1
        ).otherwise(0)
    )
)

display(
    df_entidades_cambios
    .groupBy("es_cambio")
    .count()
    .orderBy("es_cambio")
)

In [0]:
display(
    df_entidades_cambios
    .filter(
        (F.col("es_cambio") == 1) &
        F.col("nombre_anterior").isNotNull()
    )
    .select(
        "codigo_entidad",
        "fecha_corte",
        "nombre_anterior",
        "nombre_entidad",
        "nombre_normalizado"
    )
    .orderBy("codigo_entidad", "fecha_corte")
)

In [0]:
display(
    df_entidades_base
    .groupBy("codigo_entidad", "fecha_corte")
    .agg(
        F.countDistinct("nombre_normalizado")
        .alias("nombres_distintos")
    )
    .filter(F.col("nombres_distintos") > 1)
    .orderBy("codigo_entidad", "fecha_corte")
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Conservar únicamente primeras apariciones y cambios
df_versiones = (
    df_entidades_cambios
    .filter(F.col("es_cambio") == 1)
)

ventana_scd2 = (
    Window
    .partitionBy("codigo_entidad")
    .orderBy("fecha_corte")
)

df_dim_entidad_preparado = (
    df_versiones
    .withColumn(
        "fecha_inicio_vigencia",
        F.col("fecha_corte")
    )
    .withColumn(
        "siguiente_vigencia",
        F.lead("fecha_corte").over(ventana_scd2)
    )
    .withColumn(
        "fecha_fin_vigencia",
        F.when(
            F.col("siguiente_vigencia").isNotNull(),
            F.date_sub(F.col("siguiente_vigencia"), 1)
        ).otherwise(F.lit(None).cast("date"))
    )
    .withColumn(
        "es_actual",
        F.col("siguiente_vigencia").isNull()
    )
    .withColumn("id_entidad", F.col("codigo_entidad"))
    .withColumn("nit", F.lit(None).cast("string"))
    .withColumn("tipo_entidad", F.lit("POR_CLASIFICAR"))
    .withColumn("estado_entidad", F.lit("NO_VERIFICADO"))
    .withColumn("fecha_creacion", F.current_timestamp())
    .withColumn("fecha_actualizacion", F.current_timestamp())
    .select(
        "id_entidad",
        "codigo_entidad",
        "nit",
        "nombre_entidad",
        "nombre_normalizado",
        "tipo_entidad",
        "estado_entidad",
        "fecha_inicio_vigencia",
        "fecha_fin_vigencia",
        "es_actual",
        "fecha_creacion",
        "fecha_actualizacion"
    )
)

In [0]:
display(
    df_dim_entidad_preparado
    .agg(
        F.count("*").alias("total_versiones"),
        F.countDistinct("codigo_entidad").alias("entidades"),
        F.sum(
            F.when(F.col("es_actual"), 1).otherwise(0)
        ).alias("versiones_actuales"),
        F.sum(
            F.when(~F.col("es_actual"), 1).otherwise(0)
        ).alias("versiones_historicas")
    )
)

In [0]:
display(
    df_dim_entidad_preparado
    .filter(
        F.col("codigo_entidad") == "1-1-2-2-42-0-7-3-"
    )
    .select(
        "codigo_entidad",
        "nombre_entidad",
        "fecha_inicio_vigencia",
        "fecha_fin_vigencia",
        "es_actual"
    )
    .orderBy("fecha_inicio_vigencia")
)

In [0]:
display(
    df_dim_entidad_preparado
    .filter(F.col("es_actual") == True)
    .groupBy("codigo_entidad")
    .count()
    .filter(F.col("count") > 1)
)

In [0]:
%sql

SELECT COUNT(*) AS registros_actuales
FROM catalog_au.golden.dim_entidad;

In [0]:
df_dim_entidad_preparado.createOrReplaceTempView(
    "vw_dim_entidad_carga"
)

print("Vista temporal de entidades creada.")
print("Versiones preparadas:", df_dim_entidad_preparado.count())

In [0]:
%sql

MERGE INTO catalog_au.golden.dim_entidad AS destino
USING vw_dim_entidad_carga AS origen

ON destino.codigo_entidad = origen.codigo_entidad
AND destino.fecha_inicio_vigencia = origen.fecha_inicio_vigencia

WHEN NOT MATCHED THEN INSERT (
    id_entidad,
    codigo_entidad,
    nit,
    nombre_entidad,
    nombre_normalizado,
    tipo_entidad,
    estado_entidad,
    fecha_inicio_vigencia,
    fecha_fin_vigencia,
    es_actual,
    fecha_creacion,
    fecha_actualizacion
)
VALUES (
    origen.id_entidad,
    origen.codigo_entidad,
    origen.nit,
    origen.nombre_entidad,
    origen.nombre_normalizado,
    origen.tipo_entidad,
    origen.estado_entidad,
    origen.fecha_inicio_vigencia,
    origen.fecha_fin_vigencia,
    origen.es_actual,
    origen.fecha_creacion,
    origen.fecha_actualizacion
);

In [0]:
%sql

SELECT
    COUNT(*) AS total_versiones,
    COUNT(DISTINCT codigo_entidad) AS entidades,
    SUM(CASE WHEN es_actual THEN 1 ELSE 0 END)
        AS versiones_actuales,
    SUM(CASE WHEN NOT es_actual THEN 1 ELSE 0 END)
        AS versiones_historicas,
    COUNT(DISTINCT sk_entidad) AS claves_sustitutas
FROM catalog_au.golden.dim_entidad;

In [0]:
%sql

WITH control AS (
    SELECT
        codigo_entidad,
        COUNT(*) AS total_versiones,
        SUM(CASE WHEN es_actual THEN 1 ELSE 0 END)
            AS versiones_actuales,
        COUNT(DISTINCT fecha_inicio_vigencia)
            AS fechas_inicio_distintas
    FROM catalog_au.golden.dim_entidad
    GROUP BY codigo_entidad
)

SELECT *
FROM control
WHERE versiones_actuales <> 1
   OR total_versiones <> fechas_inicio_distintas;

In [0]:
%sql

SELECT
    sk_entidad,
    codigo_entidad,
    nombre_entidad,
    fecha_inicio_vigencia,
    fecha_fin_vigencia,
    es_actual
FROM catalog_au.golden.dim_entidad
WHERE codigo_entidad = '1-1-2-2-42-0-7-3-'
ORDER BY fecha_inicio_vigencia;